In [32]:
using MAT               # pour charger .mat
using LinearAlgebra
using Distributions
using Random
using Symbolics
using ModelingToolkit

In [33]:

@parameters t

# ----------------------------
# Parameters
# ----------------------------
@parameters J1 J2 D1 D2 KL Ks ω0
@parameters α1 α2 β1 β2 Pr1 Pr2

# ----------------------------
# States
# X = [θ1, ω1, Tm1, θ2, ω2, Tm2, N]
# ----------------------------
@variables θ1(t) ω1(t) Tm1(t)
@variables θ2(t) ω2(t) Tm2(t)
@variables N(t)

X = [θ1, ω1, Tm1, θ2, ω2, Tm2, N]

# ----------------------------
# Control inputs
# U = [P0_1, P0_2]
# ----------------------------
@variables P01(t) P02(t)
U = [P01, P02]

# ----------------------------
# Disturbances
# V = [PL1, PL2]
# ----------------------------
@variables PL1(t) PL2(t)
V = [PL1, PL2]

# ----------------------------
# Measurements
# Y = [PGm1, PGm2, F12]
# ----------------------------
@variables PGm1(t) PGm2(t)
F12 = KL*(θ1 - θ2)

Y = [PGm1,
    PGm2,
    F12
]
# Y = C X + E V + W
# E = [1 0;
#      0 1;
#      0 0]

f_eqs = [

    # dθ1/dt = ω1
    ω1,

    # dω1/dt
    (KL/(J1*ω0))*θ1-(D1/J1)*ω1+ (1/J1)*Tm1+ (KL/(J1*ω0))*θ2,

    # dTm1/dt
    -β1*ω1 -α1*ω0*Tm1-α1*Pr1*N,

    # dθ2/dt = ω2
    ω2,

    # dω2/dt
    (KL/(J2*ω0))*θ1-(KL/(J2*ω0))*θ2-(D2/J2)*ω2+ (1/J2)*Tm2,

    # dTm2/dt
    -β2*ω2-α2*ω0*Tm2-α2*Pr2*N,

    # dN/dt
    -Ks*(J1/(J1+J2))*ω1-Ks*(J2/(J1+J2))*ω2
]

7-element Vector{Num}:
                                                                         ω1(t)
  (KL*θ1(t)) / (J1*ω0) + (KL*θ2(t)) / (J1*ω0) + Tm1(t) / J1 + (-D1*ω1(t)) / J1
                                        -ω1(t)*β1 - Pr1*N(t)*α1 - Tm1(t)*α1*ω0
                                                                         ω2(t)
 (-D2*ω2(t)) / J2 + Tm2(t) / J2 + (-KL*θ2(t)) / (J2*ω0) + (KL*θ1(t)) / (J2*ω0)
                                        -ω2(t)*β2 - Pr2*N(t)*α2 - Tm2(t)*α2*ω0
                       (-J1*Ks*ω1(t)) / (J1 + J2) + (-J2*Ks*ω2(t)) / (J1 + J2)

In [ ]:
# State equation matrices
A = Symbolics.jacobian(f_eqs, X)
B = Symbolics.jacobian(f_eqs, U)
D = Symbolics.jacobian(f_eqs, V)

# Output matrices
C = Symbolics.jacobian(Y, X)
E = Symbolics.jacobian(Y, V)

@show typeof(A)
@show typeof(C)
@show size(A)
@show size(C)

3×2 Matrix{Num}:
 0  0
 0  0
 0  0

In [ ]:
function calcul_matrice_observabilite(A, C, n_states)
    println("1. Calcul des Jacobiennes A et C...")
    
    # Initialisation de la matrice O avec C
    O = C
    
    # Puissance courante de A (A^0, A^1, A^2...)
    A_power = I(n_states) # Matrice identité initiale (si besoin) ou on commence la boucle direct
    
    println("2. Construction de la matrice O (CA^k)...")
    
    # Terme courant CA^k
    current_term = C
    
    # Boucle pour empiler C, CA, CA^2 ... CA^(n-1)
    # On a déjà C, donc on itère n-1 fois
    for i in 1:(n_states-1)
        # On calcule le terme suivant : C * A^i
        # Pour optimiser, on peut faire : Terme_precedent * A
        # Cependant, avec Symbolics, multiplier des matrices énormes peut être lourd.
        # On recalcule A^i ou on propage. Propager est plus simple :
        
        current_term = current_term * A
        
        # Concaténation verticale (vcat)
        O = [O; current_term]
    end
    
    return O
end

calcul_matrice_observabilite (generic function with 1 method)

In [37]:
# --- EXÉCUTION ---

# Calcul
OO = calcul_matrice_observabilite(A, C, length(X))

# Affichage des dimensions pour vérifier
println("Dimensions de O : ", size(OO))

1. Calcul des Jacobiennes A et C...
2. Construction de la matrice O (CA^k)...


MethodError: MethodError: no method matching size(::Tuple{Matrix{Num}, Matrix{Num}, Matrix{Num}})
The function `size` exists, but no method is defined for this combination of argument types.
You may need to implement the `length` method or define `IteratorSize` for this type to be `SizeUnknown`.

Closest candidates are:
  size(::Tuple, !Matched::Integer)
   @ Base tuple.jl:31
  size(!Matched::HDF5.VirtualLayout)
   @ HDF5 C:\Users\lucie\.julia\packages\HDF5\Z859u\src\virtual.jl:40
  size(!Matched::Base.MethodList)
   @ Base runtime_internals.jl:1323
  ...
